#  Football Injury Risk Prediction
### Binary Classification — Will a Player Get Injured Next Season?

---

> **Algorithm:** Logistic Regression with GridSearchCV  
> **Dataset:** 800 football players with physical & lifestyle metrics  
> **Target:** `Injury_Next_Season` (0 = No Injury, 1 = Injury)

---

##  Problem Statement

Injuries are one of the most costly events in professional football — both financially and competitively. Being able to predict injury risk before the season starts allows clubs to proactively manage player load, tailor recovery programmes, and protect their most valuable assets.

This notebook builds a **Logistic Regression classifier** trained on physical, biomechanical, and lifestyle features to predict whether a player will suffer an injury in the upcoming season.

###  Notebook Structure
1. Imports & Configuration
2. Dataset Overview
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis
5. Model Building & Hyperparameter Tuning
6. Model Evaluation
7. Best Player Profile (Radar Chart)
8. Key Insights & Conclusion

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120})

print('✅ Libraries loaded.')

---
## 2. Dataset Overview

The dataset contains **800 professional football players** with physical measurements, training habits, biomechanical test scores, and lifestyle factors. The binary target `Injury_Next_Season` indicates whether the player was injured in the following season.

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Update path to match your Kaggle input directory
df_raw = pd.read_csv('/kaggle/input/data_injury.csv')

print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
display(df_raw.head())

In [ ]:
# Data types and non-null counts
print('── Data Info ────────────────────────────────────────────')
df_raw.info()

In [ ]:
# Summary statistics for numeric columns
display(df_raw.describe())

---
## 3. Data Cleaning & Preprocessing

Steps:
- **Remove duplicates** — ensure each player observation is unique
- **Drop `Position`** — categorical column dropped after initial analysis (low predictive signal after encoding considerations)
- **Fill missing numeric values** — with column means to preserve dataset size

In [ ]:
df = df_raw.copy()

# ── Remove duplicates ─────────────────────────────────────────────────────────
n_before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed: {n_before - len(df)}')

# ── Drop Position column ──────────────────────────────────────────────────────
df = df.drop(columns=['Position'])
print('Dropped: Position')

# ── Handle missing values ─────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) > 0:
    print(f'\nMissing values found in {len(missing)} column(s):')
    print(missing)
    for col in missing.index:
        if df[col].dtype != 'object':
            df[col] = df[col].fillna(df[col].mean())
    print('✅ Filled with column means.')
else:
    print('\n✅ No missing values.')

print(f'\nFinal dataset shape: {df.shape}')

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
class_counts = df['Injury_Next_Season'].value_counts()
injury_rate  = class_counts[1] / len(df)

print(f'No Injury (0): {class_counts[0]} players')
print(f'Injury    (1): {class_counts[1]} players')
print(f'Injury Rate  : {injury_rate:.1%}')

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=['#1565c0', '#d32f2f'], ax=ax, edgecolor='white', width=0.5)
ax.set_xticklabels(['No Injury (0)', 'Injury (1)'], rotation=0, fontsize=11)
ax.set_ylabel('Player Count')
ax.set_title('Target Class Distribution', fontsize=13)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4.  Exploratory Data Analysis

### 4.1 Correlation Heatmap

We examine the absolute correlation of each feature with the target to identify the strongest injury-risk signals.

In [ ]:
# Absolute correlation with target, sorted descending
corr = df.corr()['Injury_Next_Season'].abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 9))
sns.heatmap(corr.to_frame(), annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.6})
ax.set_title('|Correlation| with Injury_Next_Season', fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Top Feature Distributions by Injury Status

Visualising how the top-correlated features are distributed across injured vs non-injured players reveals the biological and lifestyle patterns driving risk.

In [ ]:
top_features = corr.drop('Injury_Next_Season').head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for label, color, name in zip([0, 1], ['#1565c0', '#d32f2f'], ['No Injury', 'Injury']):
        subset = df[df['Injury_Next_Season'] == label][feat]
        axes[i].hist(subset, bins=25, alpha=0.55, color=color, label=name, edgecolor='white')
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)

fig.suptitle('Top-6 Feature Distributions — Injury vs No Injury', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Pairplot — Top-4 Features

A pairplot reveals pairwise relationships and cluster separation between injury classes.

In [ ]:
top4 = corr.drop('Injury_Next_Season').head(4).index.tolist()
pair_df = df[top4 + ['Injury_Next_Season']].copy()
pair_df['Injury_Next_Season'] = pair_df['Injury_Next_Season'].map({0: 'No Injury', 1: 'Injury'})

pair_grid = sns.pairplot(pair_df, hue='Injury_Next_Season',
                          palette={'No Injury': '#1565c0', 'Injury': '#d32f2f'},
                          plot_kws={'alpha': 0.4}, diag_kind='kde')
pair_grid.fig.suptitle('Pairplot — Top-4 Features by Injury Status', y=1.02, fontsize=13)
plt.show()

---
## 5. Model Building & Hyperparameter Tuning

### 5.1 Train / Test Split

We use `stratify=y` to preserve the class ratio across both splits — important for imbalanced or skewed targets.

In [ ]:
X = df.drop(columns=['Injury_Next_Season'])
y = df['Injury_Next_Season']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # preserve class ratio
)

print(f'Training set : {len(X_train):,} samples ({len(X_train)/len(X):.0%})')
print(f'Test set     : {len(X_test):,}  samples ({len(X_test)/len(X):.0%})')

### 5.2 Pipeline

Wrapping `StandardScaler` and `LogisticRegression` inside a `Pipeline` ensures that:
- The scaler is fit **only** on training data within each CV fold (no data leakage)
- The full preprocessing → model chain is serialisable as a single object

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=10_000, random_state=RANDOM_STATE))
])

print(pipeline)

### 5.3 GridSearchCV — Hyperparameter Tuning

| Hyperparameter | Values Tried | Purpose |
|---|---|---|
| `C` | 0.01, 0.1, 1, 10 | Regularisation strength (lower = stronger regularisation) |
| `penalty` | l2 | L2 regularisation (Ridge-style) |
| `class_weight` | None, 'balanced' | Compensates for class imbalance |

We use **F1 score** as the optimisation metric — better than accuracy for imbalanced targets because it balances precision and recall for the minority (injury) class.

In [ ]:
param_grid = {
    'model__C':            [0.01, 0.1, 1, 10],
    'model__penalty':      ['l2'],
    'model__class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1',   # F1 preferred for imbalanced binary classification
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f'\nBest Hyperparameters : {grid.best_params_}')
print(f'Best CV F1 Score     : {grid.best_score_:.4f}')

In [ ]:
# Inspect full CV results — sorted by mean test F1
cv_results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
display(cv_results[[
    'param_model__C', 'param_model__class_weight',
    'mean_test_score', 'std_test_score', 'rank_test_score'
]].head(8).rename(columns={
    'param_model__C': 'C',
    'param_model__class_weight': 'class_weight',
    'mean_test_score': 'mean_F1',
    'std_test_score': 'std_F1',
    'rank_test_score': 'rank'
}).reset_index(drop=True))

---
## 6. Model Evaluation

### 6.1 Accuracy & Classification Report

In [ ]:
best_model = grid.best_estimator_
y_pred     = best_model.predict(X_test)

train_acc = best_model.score(X_train, y_train)
test_acc  = best_model.score(X_test,  y_test)

print(f'Train Accuracy : {train_acc:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')
print()
print('── Classification Report ─────────────────────────────────')
print(classification_report(y_test, y_pred, target_names=['No Injury', 'Injury']))

### 6.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Injury', 'Injury'])
disp.plot(cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Confusion Matrix — Injury Risk Classifier', fontsize=13)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted no injury): {tn}')
print(f'False Positives (incorrectly flagged as injury): {fp}')
print(f'False Negatives (missed actual injuries)       : {fn}')
print(f'True Positives  (correctly predicted injury)   : {tp}')

### 6.3 Feature Coefficients (Model Explainability)

Logistic Regression is inherently interpretable. The absolute value of each coefficient tells us how strongly each feature pushes the model toward predicting injury.

In [ ]:
coef_series = pd.Series(
    best_model.named_steps['model'].coef_[0],
    index=X.columns
).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#d32f2f' if v > 0 else '#1565c0' for v in coef_series.values]
coef_series.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient Value')
ax.set_title('Logistic Regression Coefficients\n(red = increases injury risk, blue = decreases)', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 7. Best Player Profile — Radar Chart

We identify the player with the highest normalised overall score across all physical and lifestyle metrics, then visualise their profile on a radar chart. This serves as a benchmark for ideal player conditioning.

In [ ]:
radar_cols = [
    'Age', 'Height_cm', 'Weight_kg', 'Training_Hours_Per_Week',
    'Matches_Played_Past_Season', 'Previous_Injury_Count',
    'Knee_Strength_Score', 'Hamstring_Flexibility', 'Reaction_Time_ms',
    'Balance_Test_Score', 'Sprint_Speed_10m_s', 'Agility_Score',
    'Sleep_Hours_Per_Night', 'Stress_Level_Score', 'Nutrition_Quality_Score',
    'Warmup_Routine_Adherence', 'BMI'
]

# Normalise all stats to [0,1] using MinMaxScaler
scaler_radar = MinMaxScaler()
df_scaled    = pd.DataFrame(scaler_radar.fit_transform(df[radar_cols]), columns=radar_cols)

# Identify player with highest average normalised score
df_temp = df.copy()
df_temp['Overall_Score'] = df_scaled.mean(axis=1)
best_idx    = df_temp['Overall_Score'].idxmax()
best_stats  = df_scaled.loc[best_idx]

print(f'Best Player index: {best_idx}')
print(f'Overall Score    : {df_temp.loc[best_idx, "Overall_Score"]:.4f}')
print(f'Injured next season: {"Yes" if df.loc[best_idx, "Injury_Next_Season"] == 1 else "No"}')

In [ ]:
labels  = radar_cols
values  = best_stats.values
N       = len(labels)
angles  = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values  = np.concatenate((values, [values[0]]))
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, values, linewidth=2.5, linestyle='solid', color='#d32f2f')
ax.fill(angles, values, alpha=0.2, color='#d32f2f')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=8)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7, color='gray')
ax.set_title('Best Player — Normalised Physical & Lifestyle Profile', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

---
## 8. Save Model & Processed Dataset

In [ ]:
# Save trained model pipeline
joblib.dump(best_model, 'Injury_classifier_model.pkl')
print('✅ Model saved → Injury_classifier_model.pkl')

# Save cleaned dataset
df.to_csv('Injury_prediction_processed_dataset.csv', index=False)
print('✅ Dataset saved → Injury_prediction_processed_dataset.csv')

---
## 9. Key Insights & Conclusion

### What the model learned:

| Finding | Evidence |
|---------|----------|
| **Previous injuries are the #1 risk factor** | Highest correlation with target; highest LR coefficient |
| **Over-training increases risk** | `Training_Hours_Per_Week` positively correlated with injury |
| **Sleep & nutrition are protective** | Negatively correlated — more sleep/better nutrition = lower risk |
| **Stress amplifies risk** | `Stress_Level_Score` shows meaningful positive correlation |
| **Balanced class weights improve recall** | The `class_weight='balanced'` setting ensures we don't miss actual injuries |

### Clinical interpretation:
- Clubs should **flag players with ≥1 prior injury** for enhanced monitoring
- **Athlete wellness programmes** (sleep, nutrition, stress management) directly reduce model-predicted injury probability
- The model can be used as a **pre-season screening tool** to stratify players into risk tiers

### Future improvements:
- Add GPS/wearable tracking data (distance covered, sprint counts, accelerations)
- Experiment with XGBoost or Random Forest for potentially higher F1
- Incorporate injury type (muscular vs ligament) for multi-class classification
- Use SHAP values for per-player explainability